In [ ]:
# Ticker's used

TICKERS = [
    # Tech
    "AAPL", "MSFT", "GOOGL", "META", "NVDA", "NFLX", "ORCL",
    # Fintech
    "V", "PYPL", "MA",
    # Consumer
    "AMZN", "TSLA", "LULU", "SBUX", "COST", "ABNB",
    # Healthcare
    "JNJ", "PFE", "UNH",
    # Energy
    "XOM", "CVX",
    # Industrial
    "BA", "UPS", "FDX",
    # Retail
    "WMT", "ETSY",
    # Crisis tape
    "PTON", "BYND", "SNAP", "ROKU",
]

print(f"Total tickers: {len(TICKERS)}")

Total tickers: 30


In [2]:
# Pull 5 years of daily OHLCV for all tickers

import yfinance as yf
import pandas as pd

START_DATE = "2017-01-01"
END_DATE = "2024-12-31"

prices = yf.download(
    tickers = TICKERS,
    start = START_DATE,
    end = END_DATE,
    interval = "1d",
    group_by = "ticker",
    auto_adjust = True,
    progress = True
)

print(f"Shape: {prices.shape}")
print(f"Date range: {prices.index.min()} to {prices.index.max()}")

[*********************100%***********************]  30 of 30 completed


Shape: (2011, 150)
Date range: 2017-01-03 00:00:00 to 2024-12-30 00:00:00


In [3]:
# Sanity-check

print(prices["AAPL"].head())
print()
print(prices["AAPL"].tail())
print()
print(f"AAPL null counts:\n{prices['AAPL'].isnull().sum()}")
print()
print(prices["SNAP"].head())
print()
print(prices["SNAP"].tail())
print()
print(f"SNAP null counts:\n{prices['SNAP'].isnull().sum()}")

Price            Open       High        Low      Close     Volume
Date                                                             
2017-01-03  26.665263  26.787306  26.425782  26.745857  115127600
2017-01-04  26.676776  26.828755  26.653749  26.715921   84472400
2017-01-05  26.692893  26.909347  26.667563  26.851780   88774400
2017-01-06  26.890928  27.208702  26.819545  27.151134  127007600
2017-01-09  27.160345  27.501145  27.158044  27.399826  134247600

Price             Open        High         Low       Close    Volume
Date                                                                
2024-12-23  253.385834  254.261043  252.072998  253.883118  40858800
2024-12-24  254.101927  256.807136  253.903002  256.797211  23234700
2024-12-26  256.787224  258.686851  256.230269  257.612701  27237100
2024-12-27  256.429191  257.294489  251.685117  254.201370  42355300
2024-12-30  250.859609  252.122713  249.387654  250.829773  35557500

AAPL null counts:
Price
Open      0
High      0
Low  

In [4]:
# Did any ticker fail to download?

null_report = {}
for t in TICKERS:
    try:
        nulls = prices[t]["Close"].isnull().sum()
        null_report[t] = nulls
    except KeyError:
        null_report[t] = "MISSING"

for t, n in sorted(null_report.items(), key = lambda x: -1 if x[1] == "MISSING" else x[1], reverse = True):
    if n != 0:
        print(f"{t}: {n} nulls")

print("\nAll other tickers: 0 nulls")

ABNB: 992 nulls
PTON: 687 nulls
BYND: 585 nulls
ROKU: 186 nulls
SNAP: 40 nulls

All other tickers: 0 nulls


In [3]:
# Save one CSV per ticker

from pathlib import Path

output_dir = Path("../data/raw/prices")
output_dir.mkdir(parents = True, exist_ok = True)

saved = 0
for t in TICKERS:
    try:
        df = prices[t].dropna(how = "all") # drop fully-empty rows (pre-IPO)
        df.to_csv(output_dir / f"{t}.csv")
        saved += 1
    except KeyError:
        print(f"Skipped {t} - not in download")

print(f"Saved {saved} ticker files to {output_dir.resolve()}")

Saved 30 ticker files to D:\Projects\risk-radar\data\raw\prices


In [6]:
# What's in the Kaggle download?

from pathlib import Path

transcripts_dir = Path("../data/raw/transcripts")
print(f"Contents of {transcripts_dir.resolve()}:\n")

for item in transcripts_dir.iterdir():
    size_mb = item.stat().st_size / 1024 / 1024
    print(f" {item.name} ({size_mb:.1f} MB)")

Contents of D:\Projects\risk-radar\data\raw\transcripts:

 motley-fool-data.pkl (847.4 MB)


In [4]:
pkl_path = Path("../data/raw/transcripts/motley-fool-data.pkl")
transcripts_raw = pd.read_pickle(pkl_path)

print(f"Type: {type(transcripts_raw).__name__}")
print(f"Total records: {len(transcripts_raw):,}")

if isinstance(transcripts_raw, pd.DataFrame):
    print(f"Shape: {transcripts_raw.shape}")
    print(f"\nColumns: {list(transcripts_raw.columns)}")
    print(f"\nFirst row:\n{transcripts_raw.iloc[0]}")
else:
    print(f"First item type: {type(transcripts_raw[0]).__name__}")
    print(f"First item:\n{transcripts_raw[0]}")

Type: DataFrame
Total records: 18,755
Shape: (18755, 5)

Columns: ['date', 'exchange', 'q', 'ticker', 'transcript']

First row:
date                                 Aug 27, 2020, 9:00 p.m. ET
exchange                                           NASDAQ: BILI
q                                                       2020-Q2
ticker                                                     BILI
transcript    Prepared Remarks:\nOperator\nGood day, and wel...
Name: 0, dtype: object


In [ ]:
# Filter Kaggle transcripts to our 30 tickers

transcripts_kaggle = transcripts_raw[transcripts_raw["ticker"].isin(TICKERS)].copy()

print(f"Filtered: {len(transcripts_kaggle):,} transcripts from {transcripts_kaggle['ticker'].nunique()} tickers")
print(f"\nTranscripts per ticker:")
print(transcripts_kaggle["ticker"].value_counts().sort_index())

missing_tickers = set(TICKERS) - set(transcripts_kaggle["ticker"].unique())
if missing_tickers:
    print(f"\n Tickers MISSING from Kaggle: {sorted(missing_tickers)}")
else:
    print(f"\n All 30 tickers present in Kaggle data")

Filtered: 470 transcripts from 30 tickers

Transcripts per ticker:
ticker
AAPL     62
ABNB      6
AMZN     39
BA        9
BYND      8
COST     10
CVX       8
ETSY     10
FDX      14
GOOGL    58
JNJ      11
LULU      8
MA       10
META     10
MSFT     10
NFLX     10
NVDA     11
ORCL     12
PFE       9
PTON     10
PYPL      9
ROKU      8
SBUX     10
SNAP     14
TSLA     52
UNH       9
UPS      10
V         9
WMT      15
XOM       9
Name: count, dtype: int64

v All 30 tickers present in Kaggle data


In [ ]:
# Investigate the high-volume tickers

print("=" * 60)
print("HIGH-VOLUME TICKER INVESTIGATION")
print("=" * 60)
for ticker in ["AAPL", "GOOGL", "TSLA", "AMZN"]:
    print("-" * 60)
    subset = transcripts_kaggle[transcripts_kaggle["ticker"] == ticker]
    print(f"{ticker} — total rows: {len(subset)}")
    print(f"Unique quarters: {subset['q'].nunique()}")
    
    # Count date column types separately (avoid the f-string)
    date_type_counts = subset["date"].apply(type).value_counts()
    print("Date column types:")
    print(date_type_counts.to_string())
    
    print("Quarter value counts (top 5):")
    print(subset["q"].value_counts().head(5).to_string())

HIGH-VOLUME TICKER INVESTIGATION
------------------------------------------------------------
AAPL — total rows: 62
Unique quarters: 14
Date column types:
date
<class 'str'>    62
Quarter value counts (top 5):
q
2020-Q2    25
2020-Q1    21
2020-Q3     5
2022-Q3     1
2019-Q3     1
------------------------------------------------------------
GOOGL — total rows: 58
Unique quarters: 15
Date column types:
date
<class 'str'>     55
<class 'list'>     3
Quarter value counts (top 5):
q
2020-Q1    15
2021-Q2    14
2020-Q4    10
2021-Q1     4
2021-Q3     3
------------------------------------------------------------
TSLA — total rows: 52
Unique quarters: 12
Date column types:
date
<class 'str'>    52
Quarter value counts (top 5):
q
2020-Q2    32
2020-Q1     5
2021-Q1     2
2020-Q4     2
2019-Q4     2
------------------------------------------------------------
AMZN — total rows: 39
Unique quarters: 12
Date column types:
date
<class 'str'>     38
<class 'list'>     1
Quarter value counts (top 5)

In [ ]:
# Quick peek at dates of transcripts we're about to drop
# dedupe, parse dates, and look at the date distribution

import pandas as pd

df = transcripts_raw[transcripts_raw["ticker"].isin(TICKERS)].copy()

def normalize_date(d):
    if isinstance(d, list):
        return d[0] if len(d) > 0 else None
    return d

df["date"] = df["date"].apply(normalize_date)
df = df.dropna(subset=["date"])
df = df.drop_duplicates(subset=["ticker", "q"], keep="first")

def parse_transcript_date(date_str):
    if not isinstance(date_str, str):
        return None
    try:
        date_part = date_str.split(",")[0] + "," + date_str.split(",")[1]
        return pd.to_datetime(date_part, format="%b %d, %Y", errors="coerce")
    except Exception:
        return None

df["date_parsed"] = df["date"].apply(parse_transcript_date)
df = df.dropna(subset=["date_parsed"])

# Show year distribution
print("Year distribution of all deduplicated Kaggle transcripts for our 30 tickers:")
print(df["date_parsed"].dt.year.value_counts().sort_index().to_string())

Year distribution of all deduplicated Kaggle transcripts for our 30 tickers:
date_parsed
2019     14
2020     20
2021    112
2022    112
2023     16


In [ ]:
# Clean and dedupe the Kaggle transcripts (2017-2021 window)

print("Starting cleanup...")
print(f"Initial row count: {len(transcripts_kaggle)}")
print(f"Columns present: {list(transcripts_kaggle.columns)}")

# Safety check — verify we're starting from raw data
if "date" not in transcripts_kaggle.columns:
    raise RuntimeError(
        f"Cell 10 expects column 'date' but found {list(transcripts_kaggle.columns)}. "
        f"This usually means Cell 10 was already run. "
        f"→ Restart kernel and run Cell 1 → 2 → 5 → 7 → 8 → 10 in order."
    )

# Step 1: Handle list-type dates
def normalize_date(d):
    if isinstance(d, list):
        return d[0] if len(d) > 0 else None
    return d

transcripts_kaggle["date"] = transcripts_kaggle["date"].apply(normalize_date)
print(f"After normalizing list-dates: {len(transcripts_kaggle)}")

# Step 2: Drop None dates
transcripts_kaggle = transcripts_kaggle.dropna(subset=["date"]).copy()
print(f"After dropping None dates: {len(transcripts_kaggle)}")

# Step 3: Deduplicate on (ticker, quarter)
transcripts_kaggle = transcripts_kaggle.drop_duplicates(subset=["ticker", "q"], keep="first").copy()
print(f"After deduping on (ticker, quarter): {len(transcripts_kaggle)}")

# Step 4: Parse date strings into datetime
def parse_transcript_date(date_str):
    if not isinstance(date_str, str):
        return None
    try:
        date_part = date_str.split(",")[0] + "," + date_str.split(",")[1]
        return pd.to_datetime(date_part, format="%b %d, %Y", errors="coerce")
    except Exception:
        return None

transcripts_kaggle["date_parsed"] = transcripts_kaggle["date"].apply(parse_transcript_date)
print(f"Date-parse success rate: {transcripts_kaggle['date_parsed'].notna().sum()} / {len(transcripts_kaggle)}")

# Step 5: Drop unparseable dates
transcripts_kaggle = transcripts_kaggle.dropna(subset=["date_parsed"]).copy()
print(f"After dropping unparseable dates: {len(transcripts_kaggle)}")

# Step 6: Filter to 2017-2023 window (EXTENDED — use all available Kaggle data)
START = pd.Timestamp("2017-01-01")
END = pd.Timestamp("2023-12-31")
transcripts_kaggle = transcripts_kaggle[(transcripts_kaggle["date_parsed"] >= START) & (transcripts_kaggle["date_parsed"] <= END)].copy()
print(f"After filtering to 2017-2023: {len(transcripts_kaggle)}")

# Step 7: Drop redundant column and rename
transcripts_kaggle = transcripts_kaggle.drop(columns=["exchange"])
transcripts_kaggle = transcripts_kaggle.rename(columns={"q": "quarter", "date": "date_raw"})

print("\n" + "=" * 60)
print("FINAL CLEAN DATASET")
print("=" * 60)
print(f"Total transcripts: {len(transcripts_kaggle)}")
print(f"Unique tickers: {transcripts_kaggle['ticker'].nunique()}")
print(f"Date range: {transcripts_kaggle['date_parsed'].min()} to {transcripts_kaggle['date_parsed'].max()}")
print(f"\nTranscripts per ticker:")
print(transcripts_kaggle["ticker"].value_counts().sort_index().to_string())

Starting cleanup...
Initial row count: 470
Columns present: ['date', 'exchange', 'q', 'ticker', 'transcript']
After normalizing list-dates: 470
After dropping None dates: 470
After deduping on (ticker, quarter): 285
Date-parse success rate: 274 / 285
After dropping unparseable dates: 274
After filtering to 2017-2023: 274

FINAL CLEAN DATASET
Total transcripts: 274
Unique tickers: 30
Date range: 2019-06-25 00:00:00 to 2023-02-02 00:00:00

Transcripts per ticker:
ticker
AAPL     14
ABNB      6
AMZN     10
BA        9
BYND      8
COST     10
CVX       8
ETSY      5
FDX      13
GOOGL    10
JNJ      10
LULU      7
MA        9
META      8
MSFT      9
NFLX      9
NVDA      8
ORCL     11
PFE       8
PTON     10
PYPL      8
ROKU      8
SBUX     10
SNAP     12
TSLA     12
UNH       9
UPS       8
V         9
WMT       7
XOM       9


In [ ]:
# Save the cleaned transcripts to data/processed/

required_cols = {"ticker", "quarter", "date_parsed", "date_raw", "transcript"}
missing = required_cols - set(transcripts_kaggle.columns)
if missing:
    raise RuntimeError(
        f"Cell 11 cannot run — missing columns: {missing}\n"
        f"Available columns: {list(transcripts_kaggle.columns)}\n"
        f"→ Restart kernel and run Cells 1 → 2 → 5 → 7 → 8 → 10 → 11 in order."
    )

# Set up output directory
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Reorder columns for readability
final_cols = ["ticker", "quarter", "date_parsed", "date_raw", "transcript"]
transcripts_final = transcripts_kaggle[final_cols].copy()

# Sort for human-readable output
transcripts_final = (
    transcripts_final
    .sort_values(["ticker", "date_parsed"])
    .reset_index(drop=True)
)

# Save
output_path = processed_dir / "transcripts.csv"
transcripts_final.to_csv(output_path, index=False)

# Confirm
print(f"Saved {len(transcripts_final)} transcripts")
print(f"Path: {output_path.resolve()}")
print(f"File size: {output_path.stat().st_size / 1024 / 1024:.1f} MB")
print(f"Date range: {transcripts_final['date_parsed'].min().date()} to {transcripts_final['date_parsed'].max().date()}")
print(f"Unique tickers: {transcripts_final['ticker'].nunique()}")
print()
print("First 3 rows:")
print(transcripts_final[["ticker", "quarter", "date_parsed"]].head(3).to_string())

Saved 274 transcripts
Path: D:\Projects\risk-radar\data\processed\transcripts.csv
File size: 14.9 MB
Date range: 2019-06-25 to 2023-02-02
Unique tickers: 30

First 3 rows:
  ticker  quarter date_parsed
0   AAPL  2019-Q3  2019-07-30
1   AAPL  2020-Q1  2020-01-28
2   AAPL  2020-Q2  2020-04-30
